# Chapter 5 — Assembling the Four-Stage Recommender

*Companion notebook for* **Modern Recommender Systems** *(Manning), Chapter 5.*

This notebook assembles and evaluates the full retrieve-and-rerank pipeline
of Sections 5.4–5.6, entirely from `recsys` package components:

| Stage | Class | Module |
|---|---|---|
| Pipeline | `FourStageRecommender` | `fourstage_recsys.pipeline` |
| Retrieval | `ANNRetrieval` | `fourstage_recsys.retrieval.ann_retrieval` |
| Filtering | `HistoryFiltering` | `fourstage_recsys.filtering.history_filtering` |
| Scoring | `CrossEncoderScoring`, `CosineScoring` | `fourstage_recsys.scoring.cross_encoder` |
| Ordering | `WeightedRanker` | `fourstage_recsys.ordering.weighted_ranker` |

**Run `ch05_similarity_learning.ipynb` first** — this notebook loads its
artifacts from `data/processed/chapter05/`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch

from recsys.fourstage_recsys.pipeline import FourStageRecommender
from recsys.fourstage_recsys.recsys_context import RecommendationContext
from recsys.fourstage_recsys.retrieval.ann_retrieval import (
    ANNRetrieval, ANNRetrievalIndex,
)
from recsys.fourstage_recsys.filtering.history_filtering import HistoryFiltering
from recsys.fourstage_recsys.scoring.cross_encoder import (
    CosineScoring, CrossEncoderReranker, CrossEncoderScoring,
    build_cross_encoder_training, train_cross_encoder,
)
from recsys.fourstage_recsys.ordering.weighted_ranker import WeightedRanker
from recsys.data.preprocessing import user_item_lists
from recsys.evaluation.retrieval import (
    evaluate_retrieval_stage, intra_list_diversity,
    segment_items_by_popularity, segment_users_by_activity,
    stratified_retrieval_recall,
)
from recsys.evaluation.metrics import ndcg_at_k, precision_at_k

np.random.seed(42)
torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ART = project_root / "data" / "processed" / "chapter05"
assert (ART / "embeddings.npz").exists(), (
    "Artifacts not found — run ch05_similarity_learning.ipynb first."
)
data = np.load(ART / "embeddings.npz")
query_embeddings = data["infonce_query"]
item_embeddings = data["infonce_cand"]
with open(ART / "mappings.json") as fh:
    mappings = json.load(fh)
item_ids = mappings["item_ids"]
item_to_idx = {mid: i for i, mid in enumerate(item_ids)}
idx_to_title = {int(k): v for k, v in mappings["idx_to_title"].items()}
anchor_idx = mappings["anchor_idx"]
num_items, emb_dim = item_embeddings.shape

read_opts = dict(dtype={"userId": str, "movieId": str})
train_df = pd.read_csv(ART / "train.csv", **read_opts)
test_df = pd.read_csv(ART / "test.csv", **read_opts)
train_items_ids = user_item_lists(train_df, item_col="movieId")
train_items_idx = user_item_lists(train_df, item_col="item_idx")
relevance_ids = test_df.groupby("userId")["movieId"].apply(set).to_dict()
print(f"{num_items:,} items, {len(train_items_ids):,} users")

## 1. The retrieval stage

`ANNRetrieval` wraps an `ANNRetrievalIndex` behind the pipeline's retrieval
interface: seed item IDs go in, `ScoredItem`s with a `similarity` score come
out, and the seeds are excluded from their own candidate pool. Same
interface as the Chapter 2 retrieval stages — milliseconds instead of
seconds.

In [ ]:
ann_index = ANNRetrievalIndex(index_type="hnsw", emb_dim=emb_dim)
ann_index.add_items(item_ids, item_embeddings)
retrieval = ANNRetrieval(ann_index, query_embeddings, item_to_idx)

anchor_id = item_ids[anchor_idx]
print(f"Retrieval spot check for {idx_to_title[anchor_idx]}:")
for item in retrieval.retrieve_similar_items([anchor_id], k=5):
    print(f"  {item.scores['similarity']:.3f}  "
          f"{idx_to_title[item_to_idx[item.item_id]]}")

## 2. Training the cross-encoder (Listing 5.8)

Feature construction is deliberately simple and derived from artifacts we
already have: item features are the candidate-tower embeddings, user
features are the mean of the user's training-item embeddings. The package
implementation includes two refinements over the minimal chapter listing —
explicit interaction features (`[u, i, u ⊙ i]`) and training negatives
sampled partly from each positive's retrieval neighbors — see the
`cross_encoder` module docstring for why both matter.

In [ ]:
item_features = torch.tensor(item_embeddings)
user_features = {
    user_id: torch.tensor(item_embeddings[idxs].mean(axis=0))
    for user_id, idxs in train_items_idx.items()
}

qn = query_embeddings / (np.linalg.norm(query_embeddings, axis=1,
                                        keepdims=True) + 1e-12)
_, neighbor_table = ann_index.index.search(
    np.ascontiguousarray(qn.astype(np.float32)), 50)        # serving distribution

users, items, labels = build_cross_encoder_training(
    train_items_idx, neighbor_table, num_items)
print(f"{len(labels):,} cross-encoder training examples")

cross_encoder = train_cross_encoder(
    CrossEncoderReranker(emb_dim, emb_dim), user_features, item_features,
    users, items, labels, epochs=5, device=DEVICE)

## 3. Assembling the pipeline (Listing 5.14)

The `FourStageRecommender` does not know or care that retrieval now uses an
HNSW index or that scoring uses a cross-encoder. Each stage was upgraded
independently behind the same interface — this is the value of the pluggable
architecture. Ordering reuses the existing `WeightedRanker`, weighted
entirely on the cross-encoder score.

In [ ]:
recommender = FourStageRecommender(
    retrieval=retrieval,
    filter=HistoryFiltering(train_items_ids),
    scorer=CrossEncoderScoring(cross_encoder, user_features, item_features,
                               item_to_idx, device=DEVICE),
    ordering=WeightedRanker(weights={"cross_encoder": 1.0}),
)

demo_user = next(iter(train_items_ids))
demo_seed = train_items_ids[demo_user][-1]
context = RecommendationContext(user_id=demo_user, seed_items=[demo_seed], k=10)
recs = recommender.recommend(context)

print(f"User {demo_user}, seeded from "
      f"'{idx_to_title[item_to_idx[demo_seed]]}':")
for item in recs:
    print(f"  {item.scores['final_score']:+.3f}  "
          f"{idx_to_title[item_to_idx[item.item_id]]}")

## 4. Optional: the transformer cross-encoder (Listings 5.9 and 5.13)

`TransformerCrossEncoder` operates on raw text — the user's history and the
item's description — and its self-attention can discover interactions no one
engineered as features. It lives in the same `cross_encoder` module and
plugs into the pipeline through `TransformerScoring`, exactly like the MLP
version.

Off by default: it requires the `transformers` library and downloads a
pretrained BERT (~420 MB), and fine-tuning on CPU is slow. Note that the
untrained scoring head produces arbitrary scores — running it before
fine-tuning only verifies the plumbing. Fine-tuning follows the same pattern
as the MLP cross-encoder; the InfoNCE loss from
`ch05_similarity_learning.ipynb` works here too.

In [ ]:
RUN_TRANSFORMER = False

if RUN_TRANSFORMER:
    from recsys.fourstage_recsys.scoring.cross_encoder import (
        TransformerCrossEncoder, TransformerScoring,
    )
    user_histories = {
        user_id: "User history: " + ", ".join(
            idx_to_title[item_to_idx[mid]] for mid in ids_[-10:])
        for user_id, ids_ in train_items_ids.items()
    }
    item_descriptions = {mid: f"Item: {idx_to_title[item_to_idx[mid]]}"
                         for mid in item_ids}
    transformer_scorer = TransformerScoring(
        TransformerCrossEncoder().to(DEVICE),
        user_histories, item_descriptions)
    recommender_t = FourStageRecommender(
        retrieval=retrieval,
        filter=HistoryFiltering(train_items_ids),
        scorer=transformer_scorer,
        ordering=WeightedRanker(weights={"cross_encoder": 1.0}),
    )
    for item in recommender_t.recommend(context):
        print(f"  {item.scores['final_score']:+.3f}  "
              f"{idx_to_title[item_to_idx[item.item_id]]}")
else:
    print("Transformer section skipped (RUN_TRANSFORMER = False).")

## 5. Evaluating the retrieval stage (Listing 5.15)

Retrieval failures are invisible in end-to-end metrics — precision@10 cannot
tell you whether a great item was never retrieved or retrieved but ranked
poorly. `evaluate_retrieval_stage` measures the stage in isolation:
retrieval recall@k (the ceiling on downstream quality) and catalog coverage
(low coverage is popularity bias showing up at the retrieval level). The
seed comes from the user's training history, never from the relevance set —
seeding from a relevant item and then excluding it from the candidates would
systematically deflate recall.

In [ ]:
evaluate_retrieval_stage(retrieval, relevance_ids, train_items_ids, k=100)

### Diversity and stratified analysis (Listing 5.16)

Two more retrieval-specific views (Section 5.6.4):

- **Intra-list diversity (ILD)** — the average pairwise cosine distance
  within each user's candidate pool. High ILD gives the scoring model
  something to choose from; low ILD means the pool is one tight cluster and
  downstream variety is already lost.
- **Stratified recall** — averages hide failure modes. Recall is broken
  down by user activity (terciles of training-history length) and by item
  popularity (top 20% of items by interactions vs. the long tail). A model
  with high average recall but near-zero long-tail recall has a popularity
  problem the overall number never shows.

In [ ]:
ild = intra_list_diversity(retrieval, relevance_ids, train_items_ids,
                           item_embeddings, item_to_idx, k=100)
print(f"Intra-list diversity@100: {ild:.3f}")

user_segments = segment_users_by_activity(train_items_ids)
item_segments = segment_items_by_popularity(
    train_df["movieId"].value_counts().to_dict(), top_frac=0.2)

stratified = stratified_retrieval_recall(
    retrieval, relevance_ids, train_items_ids,
    user_segments=user_segments, item_segments=item_segments, k=100)
pd.Series(stratified).to_frame("value")

In [ ]:
import matplotlib.pyplot as plt

user_keys = [k for k in stratified if "users" in k]
item_keys = [k for k in stratified if "items" in k]
overall = evaluate_retrieval_stage(
    retrieval, relevance_ids, train_items_ids, k=100)["retrieval_recall@100"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, keys, title in [
    (axes[0], user_keys, "By user activity"),
    (axes[1], item_keys, "By item popularity"),
]:
    labels = [k.split("(")[1].rstrip(")") for k in keys]
    ax.bar(labels, [stratified[k] for k in keys], color="steelblue")
    ax.axhline(overall, color="darkred", linestyle="--", linewidth=1,
               label=f"overall ({overall:.2f})")
    ax.set_title(title)
    ax.legend()
axes[0].set_ylabel("Retrieval recall@100")
fig.suptitle("Stratified retrieval recall: averages hide failure modes")
plt.tight_layout()
plt.show()

## 6. End-to-end: does the cross-encoder earn its latency?

The final comparison runs the full pipeline twice — once with the Chapter 2
cosine scorer and once with the cross-encoder — over the same retrieved
candidates, measuring precision@10 and NDCG@10 against each user's relevance
set, along with pipeline latency.

**Read the result against your data source.** On the real MovieLens sample
the cross-encoder comes out ahead, as in the chapter. If you are running on
a small or synthetic dataset where user behavior is fully explained by
broad-taste similarity, cosine scoring to the user's mean embedding can be
essentially the data-generating process — leaving no interaction effects for
a cross-encoder to find. A cross-encoder earns its extra latency exactly
when the data contains user-item interactions a dot product cannot express,
and real behavioral data is full of them.

In [ ]:
def evaluate_pipeline(recommender, k_top=10, max_users=500):
    precisions, ndcgs = [], []
    users = [u for u in relevance_ids if u in train_items_ids][:max_users]
    t0 = time.perf_counter()
    for user_id in users:
        ctx = RecommendationContext(
            user_id=user_id, seed_items=[train_items_ids[user_id][-1]],
            k=k_top)
        recs = [item.item_id for item in recommender.recommend(ctx)]
        precisions.append(precision_at_k(recs, relevance_ids[user_id], k_top))
        ndcgs.append(ndcg_at_k(recs, relevance_ids[user_id], k_top))
    ms = (time.perf_counter() - t0) * 1000 / len(users)
    return {"Precision@10": np.mean(precisions), "NDCG@10": np.mean(ndcgs),
            "latency (ms/request)": round(ms, 2), "users": len(users)}

cosine_pipeline = FourStageRecommender(
    retrieval=retrieval,
    filter=HistoryFiltering(train_items_ids),
    scorer=CosineScoring(user_features, item_features, item_to_idx),
    ordering=WeightedRanker(weights={"cosine": 1.0}),
)

pd.DataFrame({
    "Cosine scoring (Ch. 2 baseline)": evaluate_pipeline(cosine_pipeline),
    "Cross-encoder scoring": evaluate_pipeline(recommender),
}).T

## Summary

The pipeline interface has not changed since Chapter 2 — and that is the
point. Retrieval moved from brute-force search to an ANN-backed index;
scoring moved from cosine similarity to a trained cross-encoder; filtering
and ordering were untouched. Each upgrade happened independently behind a
stable interface, and the improvements compound. The retrieval-specific
metrics tell you *where* a quality problem lives — in the candidate pool or
in the ranking — which is what determines where the next unit of engineering
effort should go. Chapter 6 takes the next step, replacing the supervised
cross-encoder with models that bring general-purpose world knowledge to the
ranking problem.